# Macrocosm photo-z CNN — train your own version (Colab + MLflow)

**One self-contained notebook.** Everything — data loading, the **model architecture**, the
pipeline, metrics, training — is inline, so you can edit it right here in Colab.

**To make your own version:** edit the **MODEL cell** (section 2), name your run in the MLflow
cell, and run all. Compare runs in the MLflow UI.

First: **Runtime → Change runtime type → GPU**, then run top to bottom.

## 0. Setup (Colab) — install, log in, pull data
Use the Google account that's a member of the `macrocosm-lewagon` project.

In [ ]:
!pip -q install mlflow
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet /content/data/
for a in range(0, 24000, 6000):            # 4 shards = 24k cutouts (quick run; pull more for real)
    b = a + 6000
    !gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_{a:07d}_{b:07d}.npy /content/data/
DATA_DIR = '/content/data'
# real run: !gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_*.npy /content/data/

## 1. Load data into RAM
`load_set` pulls a representative, center-cropped subset (float16) from the downloaded shards.
Labels are aligned to images by row index. The model regresses `log1p(z)`.

In [ ]:
import glob, re, numpy as np, pandas as pd
SHARD = 6000

def load_set(data_dir, n=20000, crop=64, seed=0):
    z = pd.read_parquet(f'{data_dir}/catalog_v1.parquet', columns=['redshift'])['redshift'].values
    paths = sorted(glob.glob(f'{data_dir}/images_*.npy'),
                   key=lambda p: int(re.findall(r'images_(\d+)_', p)[0]))
    mm = [np.load(p, mmap_mode='r') for p in paths]
    total = min(len(z), len(paths) * SHARD)
    idx = np.sort(np.random.RandomState(seed).permutation(total)[:n])
    o = (64 - crop) // 2
    X = np.empty((len(idx), crop, crop, 5), np.float16)
    for s in np.unique(idx // SHARD):
        sel = np.where(idx // SHARD == s)[0]; rows = idx[sel] % SHARD
        X[sel] = mm[int(s)][rows][:, o:o+crop, o:o+crop, :]
    return X, z[idx].astype('float32')

N = globals().get('N', 20000)
X, z = load_set(DATA_DIR, n=N, crop=64)
y = np.log1p(z).astype('float32')
NVAL = 2000
Xtr, ytr = X[NVAL:], y[NVAL:]
Xva, zva = X[:NVAL], z[:NVAL]
print('train', Xtr.shape, '| val', Xva.shape)

## 2. 👉 THE MODEL — **edit this cell** to redesign the architecture
The example: VGG stem (cheap 64→16 downsample) + 3 lightweight **Inception** modules (multi-scale
galaxy features) → GlobalAveragePooling → a 64-d **`embedding`** (feeds the fusion head later) →
`z`. **Change anything**: more/fewer Inception blocks, deeper stem, a classification head over
redshift bins (Pasquet-style), transfer learning, different `embed_dim`. See KB MCM-A-18.

In [ ]:
from tensorflow.keras import layers as L, Model, Input

def inception(x, f1, f3r, f3, f5r, f5, fp, name):
    """4 parallel branches concatenated on channels; 1x1 bottlenecks keep it cheap."""
    b1 = L.Conv2D(f1, 1, padding='same', activation='relu', name=f'{name}_1x1')(x)
    b3 = L.Conv2D(f3r, 1, padding='same', activation='relu', name=f'{name}_3x3_reduce')(x)
    b3 = L.Conv2D(f3, 3, padding='same', activation='relu', name=f'{name}_3x3')(b3)
    b5 = L.Conv2D(f5r, 1, padding='same', activation='relu', name=f'{name}_5x5_reduce')(x)
    b5 = L.Conv2D(f5, 5, padding='same', activation='relu', name=f'{name}_5x5')(b5)
    bp = L.MaxPool2D(3, strides=1, padding='same', name=f'{name}_pool')(x)
    bp = L.Conv2D(fp, 1, padding='same', activation='relu', name=f'{name}_pool_proj')(bp)
    return L.Concatenate(axis=-1, name=f'{name}_concat')([b1, b3, b5, bp])

def build_cnn(input_shape=(64, 64, 5), embed_dim=64):
    inp = Input(shape=input_shape, name='cutout')
    x = L.Conv2D(32, 3, padding='same', activation='relu', name='stem1a')(inp)
    x = L.Conv2D(32, 3, padding='same', activation='relu', name='stem1b')(x)
    x = L.BatchNormalization(name='stem1_bn')(x); x = L.MaxPool2D(name='stem1_pool')(x)   # 32x32
    x = L.Conv2D(64, 3, padding='same', activation='relu', name='stem2')(x)
    x = L.BatchNormalization(name='stem2_bn')(x); x = L.MaxPool2D(name='stem2_pool')(x)   # 16x16
    x = inception(x, 32, 32, 48, 8, 24, 24, name='inc1'); x = L.BatchNormalization(name='inc1_bn')(x)
    x = L.MaxPool2D(name='inc1_down')(x)                                                  # 8x8
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc2'); x = L.BatchNormalization(name='inc2_bn')(x)
    x = L.MaxPool2D(name='inc2_down')(x)                                                  # 4x4
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc3'); x = L.BatchNormalization(name='inc3_bn')(x)
    x = L.GlobalAveragePooling2D(name='gap')(x)
    x = L.Dense(128, activation='relu', name='dense')(x); x = L.Dropout(0.3, name='dropout')(x)
    emb = L.Dense(embed_dim, activation='relu', name='embedding')(x)   # 64-d fusion vector
    zout = L.Dense(1, name='z')(emb)                                   # log1p(z) head
    return Model(inp, zout, name='photoz_cnn')

def build_embedder(cnn):
    """Frozen feature extractor for the fusion head (image embedding)."""
    return Model(cnn.input, cnn.get_layer('embedding').output, name='photoz_embedder')

model = build_cnn()
model.summary()
print('params:', f'{model.count_params():,}')

## 3. Pipeline + metrics (preprocess, augment, σ_MAD) — usually leave as-is
Same preprocessing as the serving backend (no train/serve skew). σ_MAD is the metric to beat
(tabular baseline = 0.0133).

In [ ]:
import tensorflow as tf

def preprocess(img, label):
    img = tf.cast(img, tf.float32); img = tf.math.asinh(img)
    m = tf.reduce_mean(img, axis=[0, 1], keepdims=True)
    s = tf.math.reduce_std(img, axis=[0, 1], keepdims=True) + 1e-6
    return (img - m) / s, label

def augment(img, label):
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_flip_left_right(img); img = tf.image.random_flip_up_down(img)
    return img, label

def make_dataset(Xa, ya, training=False, batch=128):
    ds = tf.data.Dataset.from_tensor_slices((Xa, ya))
    if training: ds = ds.shuffle(len(Xa), seed=0)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def _dz(zt, zp): zt, zp = np.asarray(zt), np.asarray(zp); return (zp - zt) / (1 + zt)
def sigma_mad(zt, zp): d = _dz(zt, zp); return float(1.4826 * np.median(np.abs(d - np.median(d))))
def outlier_rate(zt, zp): return float(np.mean(np.abs(_dz(zt, zp)) > 0.05))

class SigmaMadCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, z_true): super().__init__(); self.val_ds = val_ds; self.z_true = np.asarray(z_true)
    def on_epoch_end(self, epoch, logs=None):
        zp = np.expm1(self.model.predict(self.val_ds, verbose=0).ravel())
        sm, out = sigma_mad(self.z_true, zp), outlier_rate(self.z_true, zp)
        logs = logs if logs is not None else {}
        logs['val_sigma_MAD'] = sm; logs['val_outlier'] = out
        print(f'  -> val sigma_MAD={sm:.4f}  outlier={out*100:.1f}%')

## 4. Connect MLflow (optional)
Paste the bearer token (ask the team — not in git). Start the server first: `make mlflow-start`.
Without a token it just trains locally and skips logging.

In [ ]:
import os
MLFLOW_TOKEN = 'PASTE_MLFLOW_API_TOKEN_HERE'
USE_MLFLOW = 'PASTE' not in MLFLOW_TOKEN
if USE_MLFLOW:
    import mlflow, mlflow.tensorflow
    os.environ['MLFLOW_TRACKING_URI'] = 'https://146-148-10-86.sslip.io'
    os.environ['MLFLOW_TRACKING_TOKEN'] = MLFLOW_TOKEN
    mlflow.set_experiment('photoz-cnn')
    mlflow.tensorflow.autolog(log_models=True)
    print('MLflow: logging to', os.environ['MLFLOW_TRACKING_URI'])
else:
    print('MLflow token not set -> training without logging (fine for a quick test)')

## 5. Train + log
Name **your** run so it's easy to find in the UI.

In [ ]:
EPOCHS = globals().get('EPOCHS', 50)
RUN_NAME = 'example_vgg_inception'      # <- name YOUR run

train_ds = make_dataset(Xtr, ytr, training=True, batch=128)
val_ds   = make_dataset(Xva, np.log1p(zva), training=False, batch=256)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.Huber(delta=0.02), metrics=['mae'])
cbs = [SigmaMadCallback(val_ds, zva),
       tf.keras.callbacks.EarlyStopping('val_loss', patience=6, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-5)]

def run_training():
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs)
    zp = np.expm1(model.predict(val_ds, verbose=0).ravel())
    print('val sigma_MAD =', round(sigma_mad(zva, zp), 5), ' (tabular baseline 0.0133)')
    return zp

if USE_MLFLOW:
    import mlflow
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params({'N': len(Xtr), 'crop': 64, 'arch': 'vgg-stem+3inception',
                           'params': int(model.count_params())})
        zp = run_training()
        mlflow.log_metrics({'final_val_sigma_MAD': sigma_mad(zva, zp),
                            'final_val_outlier': outlier_rate(zva, zp)})
else:
    run_training()

## 6. Compare
Open the MLflow UI (`https://146-148-10-86.sslip.io`, GitHub-org login) and compare your run
against the example and teammates'. **Beat 0.0133 (tabular) with images**, or push toward
Pasquet's 0.0091 — more data, deeper net, a classification head, transfer learning. The winning
model's `embedding` then feeds the fusion head.